# Cascad — Hugging Face attribution baseline

This notebook runs one frozen local model at a time on Kaggle or Google Colab. Select a GPU runtime before starting. Run `qwen3-4b` first; use a fresh session for `mistral-7b` if disk space is limited.

In [1]:
import importlib
import os
import pathlib
import subprocess
import sys

REPO_URL = os.environ.get("CASCAD_REPO_URL", "https://github.com/elom354/cascad.git")
MODEL_ALIAS = os.environ.get("CASCAD_HF_MODEL", "qwen3-4b")
assert MODEL_ALIAS in {"qwen3-4b", "mistral-7b"}
assert not REPO_URL.startswith("PASTE_"), "Set REPO_URL to the published Cascad repository"

base = pathlib.Path("/kaggle/working" if pathlib.Path("/kaggle/working").exists() else "/content")
repo = base / "Cascad"
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
os.chdir(repo)
print({"repository": str(repo), "model": MODEL_ALIAS})

{'repository': '/content/Cascad', 'model': 'qwen3-4b'}


In [2]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[huggingface]"], check=True)
torch = importlib.import_module("torch")
assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook settings"
print({"torch": torch.__version__, "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)})

{'torch': '2.11.0+cu128', 'cuda': '12.8', 'gpu': 'Tesla T4'}


In [3]:
# Optional for authenticated Hub downloads: define HF_TOKEN as a notebook secret.
if "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
print("HF token configured:", bool(os.environ.get("HF_TOKEN")))

HF token configured: False


In [4]:
output = base / f"cascad-huggingface-{MODEL_ALIAS}"
command = [
    sys.executable,
    "scripts/run_huggingface_attribution.py",
    "--models", MODEL_ALIAS,
    "--quantization", "4bit",
    "--out", str(output),
]
print(" ".join(command))
subprocess.run(command, check=True)

/usr/bin/python3 scripts/run_huggingface_attribution.py --models qwen3-4b --quantization 4bit --out /content/cascad-huggingface-qwen3-4b


CalledProcessError: Command '['/usr/bin/python3', 'scripts/run_huggingface_attribution.py', '--models', 'qwen3-4b', '--quantization', '4bit', '--out', '/content/cascad-huggingface-qwen3-4b']' returned non-zero exit status 1.

In [ ]:
import json
import shutil
summary = json.loads((output / "summary.json").read_text())
print(json.dumps(summary, indent=2))
archive = shutil.make_archive(str(output), "zip", output)
print("Download this archive before closing the session:", archive)